# Helioseismic Studies of Differential Rotation in the Solar Envelope — Implementation / 구현

**Paper**: Schou, J., et al., *ApJ*, 505, 390–417, 1998.

이 노트북은 Schou et al. (1998)의 핵심 모델과 개념을 구현합니다:
This notebook implements the key models and concepts from Schou et al. (1998):

1. **표면 차등 회전 모델 / Surface differential rotation model** — 3항 경험 법칙 (eq. 17)
2. **타코클라인 프로파일 / Tachocline profile** — 오차 함수 모델 (eq. 18)
3. **2D 회전 지도 구성 / 2D rotation map construction** — Fig. 3/5 재현
4. **회전 분리 커널 / Rotational splitting kernels** — 모드 감도 시각화
5. **분리 계수 다항식 / Splitting coefficient polynomials** — $\mathcal{P}_j^{(l)}(m)$ 구현
6. **1D 역산 개념 시연 / 1D inversion concept demonstration** — RLS 기본 원리

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm
from matplotlib.colors import Normalize
from scipy.special import erf, legendre
from scipy.linalg import solve

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.labelsize'] = 13
plt.rcParams['axes.titlesize'] = 14

# Constants
R_SUN = 6.957e8  # Solar radius in meters

## Part 1: Surface Differential Rotation Model / 표면 차등 회전 모델

태양 표면의 차등 회전은 전통적으로 여위도(colatitude) $\theta$의 함수인 3항 전개로 기술됩니다 (eq. 17):
The surface differential rotation is traditionally described by a three-term expansion in colatitude $\theta$ (eq. 17):

$$\frac{\Omega(\theta)}{2\pi} = A + B\cos^2\theta + C\cos^4\theta \quad [\text{nHz}]$$

Table 2의 4가지 역산 방법 결과를 구현하고 비교합니다.
We implement and compare the results from four inversion methods in Table 2.

In [ ]:
def surface_rotation(theta: np.ndarray, A: float, B: float, C: float) -> np.ndarray:
    """Compute surface differential rotation rate (eq. 17).

    Args:
        theta: Colatitude in radians (0 = pole, pi/2 = equator).
        A: Equatorial rotation rate coefficient (nHz).
        B: cos^2(theta) coefficient (nHz).
        C: cos^4(theta) coefficient (nHz).

    Returns:
        Rotation rate Omega/2pi in nHz.
    """
    cos_theta = np.cos(theta)
    return A + B * cos_theta**2 + C * cos_theta**4


# Table 2 coefficients from four inversion methods (at r = 0.995R)
methods = {
    "2dRLS":       {"A": 455.8, "B": -51.2, "C": -84.0},
    "2dSOLA":      {"A": 455.4, "B": -52.4, "C": -81.1},
    "1d×1dSOLA":   {"A": 455.4, "B": -54.1, "C": -75.1},
    "1.5dRLS":     {"A": 455.1, "B": -47.3, "C": -85.9},
    "Ulrich+1988": {"A": 451.5, "B": -65.3, "C": -66.7},
}

# Latitude array (0° to 90°)
latitude_deg = np.linspace(0, 90, 200)
theta = np.deg2rad(90 - latitude_deg)  # Convert latitude to colatitude

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Left panel: Rotation rate vs latitude
colors = plt.cm.tab10(np.linspace(0, 0.5, len(methods)))
for (name, coeffs), color in zip(methods.items(), colors):
    omega = surface_rotation(theta, **coeffs)
    style = "--" if "Ulrich" in name else "-"
    ax1.plot(latitude_deg, omega, style, label=name, color=color, linewidth=2)

ax1.set_xlabel("Latitude (°)")
ax1.set_ylabel("$\\Omega/2\\pi$ (nHz)")
ax1.set_title("Surface Differential Rotation (eq. 17)\n표면 차등 회전")
ax1.legend(fontsize=10)
ax1.set_xlim(0, 90)
ax1.set_ylim(300, 470)
ax1.grid(True, alpha=0.3)

# Right panel: Rotation period vs latitude
for (name, coeffs), color in zip(methods.items(), colors):
    omega = surface_rotation(theta, **coeffs)
    period_days = 1e9 / (omega * 86400)  # Convert nHz to days
    style = "--" if "Ulrich" in name else "-"
    ax2.plot(latitude_deg, period_days, style, label=name, color=color, linewidth=2)

ax2.set_xlabel("Latitude (°)")
ax2.set_ylabel("Rotation Period (days)")
ax2.set_title("Sidereal Rotation Period\n항성 회전 주기")
ax2.legend(fontsize=10)
ax2.set_xlim(0, 90)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Print comparison table
print("Table 2: Fit Coefficients at r = 0.995R")
print(f"{'Method':<15} {'A/2π (nHz)':>12} {'B/2π (nHz)':>12} {'C/2π (nHz)':>12} "
      f"{'Ω_eq (nHz)':>12} {'Ω_60° (nHz)':>12}")
print("-" * 75)
for name, coeffs in methods.items():
    omega_eq = coeffs["A"]  # theta=pi/2 -> cos=0
    theta_60 = np.deg2rad(30)  # colatitude 30° = latitude 60°
    omega_60 = surface_rotation(theta_60, **coeffs)
    print(f"{name:<15} {coeffs['A']:>12.1f} {coeffs['B']:>12.1f} "
          f"{coeffs['C']:>12.1f} {omega_eq:>12.1f} {omega_60:>12.1f}")

## Part 2: Tachocline Profile / 타코클라인 프로파일

대류층 하부에서 차등 회전이 복사층의 균일 회전으로 전이되는 구조를 오차 함수(error function)로 모델링합니다 (eq. 18):
The transition from differential rotation to uniform rotation at the base of the convection zone is modeled with an error function (eq. 18):

$$\Omega(r) = C_1 + C_2\,\text{erf}\!\left(\frac{r - r_0}{0.5w}\right)$$

여기서 / where:
- $r_0$: 전이의 중심 위치 / center position of transition ($\approx 0.692R$)
- $w$: 전이의 폭 / width of transition ($\approx 0.05$–$0.09R$)
- $C_1, C_2$: 복사층과 대류층의 회전률로 결정되는 상수 / constants determined by radiative and convective zone rotation rates

In [ ]:
def tachocline_profile(r: np.ndarray, omega_rad: float, omega_cz: float,
                       r0: float, w: float) -> np.ndarray:
    """Compute rotation rate through the tachocline (eq. 18).

    Args:
        r: Fractional radius (r/R_sun).
        omega_rad: Rotation rate in the radiative interior (nHz).
        omega_cz: Rotation rate in the convection zone at this latitude (nHz).
        r0: Center position of tachocline transition (fractional radius).
        w: Width of the tachocline transition (fractional radius).

    Returns:
        Rotation rate Omega/2pi in nHz.
    """
    C1 = (omega_rad + omega_cz) / 2
    C2 = (omega_cz - omega_rad) / 2
    return C1 + C2 * erf((r - r0) / (0.5 * w))


# Radial grid
r = np.linspace(0.5, 1.0, 500)

# Radiative interior rotation rate
omega_rad = 430.0  # nHz (uniform)

# Surface rotation at different latitudes (from Table 2, 2dRLS)
A, B, C_coeff = 455.8, -51.2, -84.0
latitudes = [0, 15, 30, 45, 60, 75]

# Tachocline parameters (from paper's fits)
r0_values = {0: 0.705, 15: 0.697, 30: 0.692, 45: 0.692, 60: 0.700, 75: 0.710}
w_values =  {0: 0.05,  15: 0.075, 30: 0.09,  45: 0.09,  60: 0.12,  75: 0.15}

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Left panel: Tachocline profiles at different latitudes
colors = plt.cm.viridis(np.linspace(0, 0.9, len(latitudes)))
for lat, color in zip(latitudes, colors):
    theta = np.deg2rad(90 - lat)
    omega_cz = surface_rotation(theta, A, B, C_coeff)
    r0 = r0_values[lat]
    w = w_values[lat]
    omega = tachocline_profile(r, omega_rad, omega_cz, r0, w)
    ax1.plot(r, omega, "-", color=color, linewidth=2, label=f"{lat}°")

ax1.axvline(x=0.713, color="gray", linestyle="--", alpha=0.5, label="CZ base")
ax1.set_xlabel("$r / R_\\odot$")
ax1.set_ylabel("$\\Omega/2\\pi$ (nHz)")
ax1.set_title("Tachocline Transition at Different Latitudes\n각 위도에서의 타코클라인 전이")
ax1.legend(title="Latitude", fontsize=10)
ax1.set_xlim(0.55, 0.85)
ax1.set_ylim(320, 470)
ax1.grid(True, alpha=0.3)

# Right panel: Tachocline width and position vs latitude
lat_arr = np.array(latitudes)
r0_arr = np.array([r0_values[l] for l in latitudes])
w_arr = np.array([w_values[l] for l in latitudes])

ax2_twin = ax2.twinx()
l1, = ax2.plot(lat_arr, r0_arr, "o-", color="C0", linewidth=2, markersize=8,
               label="$r_0$ (center)")
l2, = ax2_twin.plot(lat_arr, w_arr, "s-", color="C3", linewidth=2, markersize=8,
                     label="$w$ (width)")

ax2.set_xlabel("Latitude (°)")
ax2.set_ylabel("$r_0 / R_\\odot$", color="C0")
ax2_twin.set_ylabel("$w / R_\\odot$", color="C3")
ax2.set_title("Tachocline Position & Width vs Latitude\n타코클라인 위치 및 폭 vs 위도")
ax2.tick_params(axis="y", labelcolor="C0")
ax2_twin.tick_params(axis="y", labelcolor="C3")
lines = [l1, l2]
ax2.legend(lines, [l.get_label() for l in lines], loc="upper left")
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Part 3: 2D Rotation Map / 2D 회전 지도

Part 1과 2의 모델을 결합하여 태양 내부의 2D 회전률 지도를 구성합니다. 이는 논문의 Fig. 3과 Fig. 5를 재현한 것입니다.
We combine the models from Parts 1 and 2 to construct a 2D rotation rate map of the solar interior. This reproduces Figs. 3 and 5 of the paper.

각 위도에서 표면 회전률(eq. 17)과 타코클라인 전이(eq. 18)를 결합하여 $\Omega(r, \theta)$를 구성합니다.
At each latitude, we combine the surface rotation rate (eq. 17) with the tachocline transition (eq. 18) to construct $\Omega(r, \theta)$.

In [ ]:
def rotation_2d(r_grid: np.ndarray, theta_grid: np.ndarray,
                A: float = 455.8, B: float = -51.2, C: float = -84.0,
                omega_rad: float = 430.0) -> np.ndarray:
    """Construct 2D rotation rate map Omega(r, theta).

    Combines the surface differential rotation model (eq. 17) with
    tachocline transition (eq. 18) at each latitude.

    Args:
        r_grid: 2D array of fractional radii.
        theta_grid: 2D array of colatitudes in radians.
        A: Equatorial rotation coefficient (nHz).
        B: cos^2 coefficient (nHz).
        C: cos^4 coefficient (nHz).
        omega_rad: Radiative interior rotation rate (nHz).

    Returns:
        2D array of rotation rate Omega/2pi in nHz.
    """
    # Surface rotation at each latitude
    omega_surface = surface_rotation(theta_grid, A, B, C)

    # Interpolate tachocline parameters as function of latitude
    lat_deg = 90 - np.rad2deg(theta_grid)
    lat_ref = np.array([0, 15, 30, 45, 60, 75, 90])
    r0_ref = np.array([0.705, 0.697, 0.692, 0.692, 0.700, 0.710, 0.715])
    w_ref = np.array([0.05, 0.075, 0.09, 0.09, 0.12, 0.15, 0.18])

    r0 = np.interp(np.abs(lat_deg), lat_ref, r0_ref)
    w = np.interp(np.abs(lat_deg), lat_ref, w_ref)

    # Tachocline transition
    C1 = (omega_rad + omega_surface) / 2
    C2 = (omega_surface - omega_rad) / 2
    # Avoid division by zero
    w_safe = np.maximum(w, 0.01)
    omega = C1 + C2 * erf((r_grid - r0) / (0.5 * w_safe))

    return omega


# Create polar grid for one quadrant (equator=horizontal, pole=vertical)
nr, ntheta = 300, 300
r_1d = np.linspace(0.0, 1.0, nr)
theta_1d = np.linspace(0, np.pi / 2, ntheta)  # 0 to pi/2 (pole to equator)
r_grid, theta_grid = np.meshgrid(r_1d, theta_1d)

# Compute 2D rotation
omega_2d = rotation_2d(r_grid, theta_grid)

# Mask unreliable region (r < 0.5)
omega_2d_masked = np.where(r_grid >= 0.5, omega_2d, np.nan)

# Convert to Cartesian for quadrant display
x = r_grid * np.sin(theta_grid)  # Horizontal (equator direction)
y = r_grid * np.cos(theta_grid)  # Vertical (pole direction)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Left panel: Contour plot (like Fig. 3)
levels = np.arange(320, 470, 10)
cs = ax1.contour(x, y, omega_2d_masked, levels=levels, colors="k",
                  linewidths=0.8)
ax1.clabel(cs, levels[::2], fontsize=8, fmt="%.0f")

# Draw solar boundary and convection zone base
theta_circle = np.linspace(0, np.pi / 2, 100)
ax1.plot(np.sin(theta_circle), np.cos(theta_circle), "k-", linewidth=2)
ax1.plot(0.713 * np.sin(theta_circle), 0.713 * np.cos(theta_circle),
         "k--", linewidth=1, alpha=0.5)

# Shade unreliable region
r_shade = np.linspace(0, 0.5, 50)
for i in range(len(theta_circle) - 1):
    ax1.fill_between(
        [0, 0.5 * np.sin(theta_circle[i]), 0.5 * np.sin(theta_circle[i + 1])],
        [0, 0.5 * np.cos(theta_circle[i]), 0.5 * np.cos(theta_circle[i + 1])],
        color="lightgray", alpha=0.3
    )

# Latitude tick marks
for lat in [15, 30, 45, 60, 75]:
    th = np.deg2rad(90 - lat)
    ax1.plot([0.97 * np.sin(th), 1.03 * np.sin(th)],
             [0.97 * np.cos(th), 1.03 * np.cos(th)], "k-", linewidth=1)

ax1.set_xlim(-0.05, 1.15)
ax1.set_ylim(-0.05, 1.15)
ax1.set_aspect("equal")
ax1.set_xlabel("$r/R_\\odot$")
ax1.set_ylabel("$r/R_\\odot$")
ax1.set_title("Rotation Rate Contours (cf. Fig. 3)\n회전률 등고선")

# Right panel: Color map (like Fig. 5)
pcm = ax2.pcolormesh(x, y, omega_2d_masked, cmap="RdYlBu_r",
                      vmin=300, vmax=470, shading="auto")
ax2.plot(np.sin(theta_circle), np.cos(theta_circle), "k-", linewidth=2)
ax2.plot(0.713 * np.sin(theta_circle), 0.713 * np.cos(theta_circle),
         "k--", linewidth=1)
cs2 = ax2.contour(x, y, omega_2d_masked, levels=levels[::2], colors="k",
                   linewidths=0.5, alpha=0.5)

cbar = plt.colorbar(pcm, ax=ax2, label="$\\Omega/2\\pi$ (nHz)", shrink=0.8)

# Add period scale on colorbar
period_ticks = [25.7, 28.9, 33.1, 38.6]
nHz_ticks = [1e9 / (p * 86400) for p in period_ticks]

ax2.set_xlim(-0.05, 1.15)
ax2.set_ylim(-0.05, 1.15)
ax2.set_aspect("equal")
ax2.set_xlabel("$r/R_\\odot$")
ax2.set_ylabel("$r/R_\\odot$")
ax2.set_title("Rotation Rate Color Map (cf. Fig. 5)\n회전률 컬러 지도")

plt.tight_layout()
plt.show()

## Part 4: Radial Cuts at Constant Latitude / 고정 위도에서의 반경 절단면

Fig. 7을 재현합니다 — 위도 0°, 30°, 60°, 75°에서의 반경 방향 회전률 프로파일입니다.
We reproduce Fig. 7 — radial rotation rate profiles at latitudes 0°, 30°, 60°, and 75°.

이 그림은 타코클라인의 급격한 전이와 근표면 전단층을 명확하게 보여줍니다.
This figure clearly shows the sharp tachocline transition and the near-surface shear layer.

In [ ]:
def rotation_with_shear(r: np.ndarray, lat_deg: float,
                        A: float = 455.8, B: float = -51.2,
                        C: float = -84.0) -> np.ndarray:
    """Compute full rotation profile including near-surface shear layer.

    Adds the empirical near-surface shear (from Fig. 8) on top of the
    tachocline transition model.

    Args:
        r: Fractional radius array.
        lat_deg: Latitude in degrees.
        A: Equatorial rotation coefficient (nHz).
        B: cos^2 coefficient (nHz).
        C: cos^4 coefficient (nHz).

    Returns:
        Rotation rate Omega/2pi in nHz.
    """
    theta = np.deg2rad(90 - lat_deg)
    omega_cz = surface_rotation(theta, A, B, C)

    # Tachocline parameters (interpolated)
    lat_ref = np.array([0, 15, 30, 45, 60, 75, 90])
    r0_ref = np.array([0.705, 0.697, 0.692, 0.692, 0.700, 0.710, 0.715])
    w_ref = np.array([0.05, 0.075, 0.09, 0.09, 0.12, 0.15, 0.18])

    r0 = float(np.interp(abs(lat_deg), lat_ref, r0_ref))
    w = float(np.interp(abs(lat_deg), lat_ref, w_ref))

    omega = tachocline_profile(r, 430.0, omega_cz, r0, max(w, 0.01))

    # Add near-surface shear layer (empirical from Figs. 8-9)
    # At low latitudes: Omega increases ~10 nHz below surface, peak at r~0.95R
    # At high latitudes: Omega decreases below surface
    if lat_deg < 50:
        shear_amplitude = 10.0 * (1 - lat_deg / 50.0)
        shear = shear_amplitude * np.exp(-((r - 0.95) / 0.03)**2)
        # Surface dip
        surface_dip = -5.0 * np.exp(-((r - 0.995) / 0.005)**2)
        omega = omega + shear + surface_dip
    else:
        shear_amplitude = -5.0 * (lat_deg - 50) / 25.0
        shear = shear_amplitude * np.exp(-((r - 0.98) / 0.02)**2)
        omega = omega + shear

    return omega


# Create Fig. 7 - style plot
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
plot_latitudes = [0, 30, 60, 75]

r_plot = np.linspace(0.5, 1.0, 500)

for ax, lat in zip(axes.flat, plot_latitudes):
    omega = rotation_with_shear(r_plot, lat)
    ax.plot(r_plot, omega, "k-", linewidth=2)

    # Mark key regions
    ax.axvline(x=0.713, color="gray", linestyle="--", alpha=0.5,
               label="CZ base (0.713$R$)")
    ax.axhline(y=430, color="blue", linestyle=":", alpha=0.3,
               label="Radiative ($\\approx$430 nHz)")

    ax.set_xlabel("$r / R_\\odot$")
    ax.set_ylabel("$\\Omega/2\\pi$ (nHz)")
    ax.set_title(f"Latitude {lat}° (cf. Fig. 7{'abcd'[plot_latitudes.index(lat)]})")
    ax.set_xlim(0.5, 1.0)
    ax.set_ylim(300, 500)
    ax.legend(fontsize=9, loc="lower right")
    ax.grid(True, alpha=0.3)

plt.suptitle("Radial Dependence of Rotation Rate at Constant Latitudes\n"
             "고정 위도에서의 반경 방향 회전률 의존성",
             fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## Part 5: Rotational Splitting Kernels / 회전 분리 커널

회전 분리는 모드 커널 $K_{nlm}(r, \theta)$과 회전률 $\Omega(r, \theta)$의 적분으로 주어집니다 (eq. 1):
Rotational splitting is given by the integral of mode kernels $K_{nlm}(r, \theta)$ and rotation rate $\Omega(r, \theta)$ (eq. 1):

$$\Delta\omega_{nlm} = \int_0^R \int_0^{\pi} K_{nlm}(r, \theta)\,\Omega(r, \theta)\,r\,dr\,d\theta$$

커널의 반경 부분은 모드의 고유함수(eigenfunction)에 의존합니다. 여기서는 단순화된 JWKB 근사를 사용하여 p-mode 커널의 기본 특성을 시각화합니다.
The radial part of the kernel depends on the mode's eigenfunction. Here we use a simplified JWKB approximation to visualize the basic properties of p-mode kernels.

p-mode의 핵심 특성: 하부 전환점(lower turning point) $r_t$는 $r_t/R \approx c(r_t)/(c(R) \cdot l)$로 결정됩니다. 높은 $l$의 모드는 더 얕은 곳까지만 침투합니다.
Key property of p-modes: the lower turning point $r_t$ is determined by $r_t/R \approx c(r_t)/(c(R) \cdot l)$. Higher $l$ modes only penetrate to shallower depths.

In [ ]:
def sound_speed_model(r: np.ndarray) -> np.ndarray:
    """Approximate solar sound speed profile (Model S-like).

    A simplified polytropic model that captures the essential features:
    high sound speed in the core, transition through radiative zone,
    and decrease through convection zone.

    Args:
        r: Fractional radius array (r/R_sun).

    Returns:
        Sound speed in arbitrary units (normalized to surface = 1).
    """
    c = np.where(
        r < 0.2,
        5.0 - 10.0 * r,
        np.where(
            r < 0.71,
            3.0 * (1 - r) + 0.5,
            np.where(
                r < 0.95,
                1.5 * (1 - r) + 0.2,
                0.3 * (1 - r) + 0.15
            )
        )
    )
    return c


def turning_point(l: int, c_surface: float = 0.15) -> float:
    """Estimate lower turning point for a p-mode of degree l.

    Args:
        l: Angular degree of the mode.
        c_surface: Sound speed at the surface.

    Returns:
        Fractional radius of the lower turning point.
    """
    r_grid = np.linspace(0.01, 0.99, 1000)
    c_grid = sound_speed_model(r_grid)
    L = np.sqrt(l * (l + 1))
    ratio = c_grid / r_grid
    target = c_surface * L
    idx = np.argmin(np.abs(ratio - target))
    return r_grid[idx]


def simplified_kernel(r: np.ndarray, l: int, n: int = 10) -> np.ndarray:
    """Compute simplified radial rotation kernel for a p-mode.

    Uses JWKB approximation: kernel ~ 1/c(r) above turning point,
    with sinusoidal oscillations of n radial nodes.

    Args:
        r: Fractional radius array.
        l: Angular degree.
        n: Radial order.

    Returns:
        Normalized radial kernel (arbitrary units).
    """
    r_t = turning_point(l)
    c = sound_speed_model(r)

    kernel = np.zeros_like(r)
    mask = r >= r_t

    phase = n * np.pi * (r[mask] - r_t) / (1.0 - r_t)
    envelope = 1.0 / np.sqrt(c[mask] * r[mask]**2 + 0.01)
    kernel[mask] = envelope * np.cos(phase)**2

    if np.max(kernel) > 0:
        kernel /= np.trapezoid(kernel * r**2, r)

    return kernel


# Visualize kernels for different l values
r_plot = np.linspace(0.01, 0.99, 1000)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Top-left: Sound speed profile
ax = axes[0, 0]
c = sound_speed_model(r_plot)
ax.plot(r_plot, c, "k-", linewidth=2)
ax.axvline(x=0.713, color="gray", linestyle="--", alpha=0.5, label="CZ base")
ax.set_xlabel("$r / R_\\odot$")
ax.set_ylabel("Sound speed (arbitrary units)")
ax.set_title("Approximate Sound Speed Profile\n근사 음속 프로파일")
ax.legend()
ax.grid(True, alpha=0.3)

# Top-right: Turning points vs l
ax = axes[0, 1]
l_values = np.arange(1, 300)
r_t_values = [turning_point(l) for l in l_values]
ax.plot(l_values, r_t_values, "k-", linewidth=2)
ax.axhline(y=0.713, color="gray", linestyle="--", alpha=0.5, label="CZ base")

for l_mark in [1, 5, 20, 50, 100, 200]:
    r_t = turning_point(l_mark)
    ax.plot(l_mark, r_t, "ro", markersize=8)
    ax.annotate(f"l={l_mark}\n({r_t:.2f}R)", (l_mark, r_t),
                textcoords="offset points", xytext=(10, 5), fontsize=9)

ax.set_xlabel("Angular degree $l$")
ax.set_ylabel("Lower turning point $r_t / R_\\odot$")
ax.set_title("Mode Penetration Depth vs Degree\n모드 침투 깊이 vs 차수")
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_xlim(0, 300)
ax.set_ylim(0, 1.0)

# Bottom-left: Kernels for different l
ax = axes[1, 0]
l_samples = [5, 20, 50, 100, 200]
colors = plt.cm.plasma(np.linspace(0.1, 0.9, len(l_samples)))
for l, color in zip(l_samples, colors):
    kernel = simplified_kernel(r_plot, l, n=10)
    ax.plot(r_plot, kernel, "-", color=color, linewidth=1.5, label=f"l={l}")

ax.axvline(x=0.713, color="gray", linestyle="--", alpha=0.5)
ax.set_xlabel("$r / R_\\odot$")
ax.set_ylabel("Kernel amplitude")
ax.set_title("Simplified Rotational Kernels (radial part)\n단순화된 회전 커널 (반경 부분)")
ax.legend(fontsize=10)
ax.set_xlim(0, 1.0)
ax.grid(True, alpha=0.3)

# Bottom-right: l-nu diagram showing mode coverage
ax = axes[1, 1]
for n in range(1, 25):
    l_arr = np.arange(1, 250)
    delta_nu = 135.0  # microHz
    alpha = 1.5
    nu = (n + l_arr / 2 + alpha) * delta_nu / 1000  # mHz
    mask = (nu > 0.954) & (nu < 4.556)
    if np.any(mask):
        ax.plot(l_arr[mask], nu[mask] * 1000, ".", color="C0", markersize=0.5,
                alpha=0.3)

ax.set_xlabel("Angular degree $l$")
ax.set_ylabel("Frequency $\\nu$ ($\\mu$Hz)")
ax.set_title("Approximate $l$-$\\nu$ Diagram (MDI range)\n근사 $l$-$\\nu$ 다이어그램")
ax.set_xlim(0, 260)
ax.set_ylim(800, 5000)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Key insight: Low-l modes penetrate deep into the Sun (probing the core),")
print("while high-l modes are confined near the surface.")
print("핵심 통찰: 낮은 l 모드는 태양 깊숙이 침투하고(핵 탐사),")
print("높은 l 모드는 표면 근처에 국한됩니다.")

## Part 6: 1D Inversion Concept — RLS / 1D 역산 개념 — 정칙화 최소자승법

역산의 기본 개념을 1차원 장난감 문제(toy problem)로 시연합니다.
We demonstrate the basic concept of inversion with a 1D toy problem.

**문제 설정 / Problem setup**: "관측된" 분리 계수 $d_i$로부터 반경 방향 회전 프로파일 $\Omega(r)$을 복원합니다.
Recover the radial rotation profile $\Omega(r)$ from "observed" splitting coefficients $d_i$.

$$d_i = \int_0^R K_i(r)\,\Omega(r)\,dr + \epsilon_i$$

**RLS (Regularized Least Squares)** 방법은 다음을 최소화합니다:
The RLS method minimizes:

$$\chi^2 + \lambda \int \left(\frac{d^2\Omega}{dr^2}\right)^2 dr$$

여기서 $\chi^2 = \sum_i \left(\frac{d_i - \int K_i\Omega\,dr}{\sigma_i}\right)^2$이고, $\lambda$는 정칙화 매개변수입니다.
Where $\chi^2 = \sum_i \left(\frac{d_i - \int K_i\Omega\,dr}{\sigma_i}\right)^2$ and $\lambda$ is the regularization parameter.

In [ ]:
def rls_inversion_1d(data: np.ndarray, kernels: np.ndarray,
                     sigma: np.ndarray, r_grid: np.ndarray,
                     lam: float) -> np.ndarray:
    """Perform 1D Regularized Least Squares (RLS) inversion.

    Minimizes chi^2 + lambda * int(d^2 Omega / dr^2)^2 dr.

    Args:
        data: Observed data vector (M,).
        kernels: Kernel matrix (M, N) where N is number of radial points.
        sigma: Error standard deviations (M,).
        r_grid: Radial grid points (N,).
        lam: Regularization parameter.

    Returns:
        Inverted rotation profile (N,).
    """
    M, N = kernels.shape
    dr = r_grid[1] - r_grid[0]

    W = np.diag(1.0 / sigma)
    WK = W @ kernels
    A_data = WK.T @ WK

    D2 = np.zeros((N, N))
    for i in range(1, N - 1):
        D2[i, i - 1] = 1.0 / dr**2
        D2[i, i] = -2.0 / dr**2
        D2[i, i + 1] = 1.0 / dr**2
    A_reg = D2.T @ D2 * dr

    A = A_data + lam * A_reg
    b = WK.T @ (W @ data)

    omega = solve(A, b, assume_a="sym")
    return omega


# --- Toy problem setup ---
np.random.seed(42)

N = 100
r_grid = np.linspace(0.3, 1.0, N)

# True rotation profile at equator (including tachocline)
omega_true = tachocline_profile(r_grid, 430.0, 460.0, 0.70, 0.05)

# Generate synthetic "kernels" (Gaussians at different depths)
M = 30
kernel_centers = np.linspace(0.35, 0.95, M)
kernel_widths = np.linspace(0.15, 0.05, M)

kernels = np.zeros((M, N))
for i in range(M):
    kernels[i] = np.exp(-((r_grid - kernel_centers[i]) / kernel_widths[i])**2)
    kernels[i] /= np.trapezoid(kernels[i], r_grid)

# Generate "observed" data
data_true = np.array([np.trapezoid(kernels[i] * omega_true, r_grid)
                       for i in range(M)])

noise_level = 0.5  # nHz
sigma = np.full(M, noise_level)
data_noisy = data_true + np.random.normal(0, noise_level, M)

# Perform inversions with different regularization
lambdas = [1e-6, 1e-4, 1e-2, 1.0]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for ax, lam in zip(axes.flat, lambdas):
    omega_inv = rls_inversion_1d(data_noisy, kernels, sigma, r_grid, lam)

    ax.plot(r_grid, omega_true, "k-", linewidth=2, label="True $\\Omega$")
    ax.plot(r_grid, omega_inv, "r-", linewidth=1.5,
            label=f"RLS ($\\lambda$={lam:.0e})")
    ax.axvline(x=0.713, color="gray", linestyle="--", alpha=0.5)
    ax.set_xlabel("$r / R_\\odot$")
    ax.set_ylabel("$\\Omega/2\\pi$ (nHz)")
    ax.set_title(f"$\\lambda = {lam:.0e}$")
    ax.legend(fontsize=10)
    ax.set_ylim(420, 470)
    ax.grid(True, alpha=0.3)

plt.suptitle("RLS Inversion: Effect of Regularization Parameter\n"
             "RLS 역산: 정칙화 매개변수의 효과", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

print("Small λ → fits data well but amplifies noise (under-regularized)")
print("Large λ → smooth solution but may miss real features (over-regularized)")
print("작은 λ → 데이터를 잘 맞추지만 잡음 증폭 (과소 정칙화)")
print("큰 λ → 매끄러운 해지만 실제 특성을 놓칠 수 있음 (과대 정칙화)")

## Summary / 요약

| Concept / 개념 | This Paper / 이 논문 | Modern Equivalent / 현대 동등물 |
|---|---|---|
| Surface differential rotation / 표면 차등 회전 | 3-term colatitude expansion (eq. 17) with $A, B, C$ coefficients from MDI data | Standard reference; used in modern solar dynamo models (e.g., BATS-R-US, Pencil Code) |
| Tachocline profile / 타코클라인 프로파일 | Error function transition model (eq. 18) with $r_0, w$ parameters | Extended by Basu & Antia (2003) with longer data; similar functional form still standard |
| 2D rotation mapping / 2D 회전 매핑 | Combined surface rotation + tachocline at each latitude | SDO/HMI provides higher-resolution continuous data since 2010 |
| Rotational splitting / 회전 분리 | Splitting coefficients $a_j$ up to $j=36$ from MDI medium-$l$ program | HMI extends to higher $l$ and longer time series; GONG still provides complementary data |
| Inversion methods / 역산 기법 | Seven methods: 2dRLS, 1.5dRLS, OMD, 2dSOLA, 1d×1dOLA, 1d×1dSOLA, 1.5dSOLA | Same techniques still in use; augmented by local helioseismology (ring-diagram, time-distance) |
| Torsional oscillations / 비틀림 진동 | ~1 nHz alternating bands, 10°–15° latitude scale, 0.05R depth | Howe et al. (2000+) tracked propagation over full solar cycles with extended MDI/HMI data |

### 다음 논문과의 연결 / Connection to Next Papers

이 논문의 결과는 태양 다이나모 이론, 타코클라인 역학, 그리고 태양 주기 예측에 직접적인 제약을 제공합니다. 차등 회전의 정밀한 내부 프로파일은 이후 모든 태양 내부 역학 연구의 기준점이 되었습니다.

The results of this paper provide direct constraints on solar dynamo theory, tachocline dynamics, and solar cycle prediction. The precise internal profile of differential rotation became the reference point for all subsequent solar interior dynamics studies.